# 第13周 · GNN 与预训练概念；论文阅读与复现启动

**本周结束时你能做到**：用邻接矩阵描述一张图，手算一次消息传递；从零写出一层 GCN（它就是一次归一化邻接矩阵乘法），并验证它对节点重新编号是等变的；用 PyTorch Geometric 在 Cora 引用网络上做节点分类，说清"利用图结构"带来多少提升、层数太多为什么会过平滑；说出掩码语言模型（BERT）与自回归模型（GPT）两种预训练目标的区别，用冻结的 DistilBERT 句子嵌入做线性探针；用三遍阅读法读论文，并为复现任务选题。

本周是一个转折点：前 12 周你一直在**从零训练自己的模型**；从本周起，你开始**站在别人的工作上**——用别人写好的图神经网络库、用别人预训练好的模型、复现别人论文里的一张图。

S1–S2 改编自 PyTorch Geometric 官方 Colab 教程（[Colab Notebooks 列表](https://pytorch-geometric.readthedocs.io/en/latest/get_started/colabs.html)，MIT License）的前两篇：*Introduction: Hands-on Graph Neural Networks*（Karate Club）与 *Node Classification with Graph Neural Networks*（Cora：先 MLP 基线、再 GCN）。GCN 出自 Kipf & Welling, *Semi-Supervised Classification with Graph Convolutional Networks*, ICLR 2017（[arXiv:1609.02907](https://arxiv.org/abs/1609.02907)）；Kipf 的博客 [Graph Convolutional Networks](https://tkipf.github.io/graph-convolutional-networks/)（2016）用的也正是 Karate Club。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：PyTorch Geometric（2.3 起是纯 Python 包，不需要 pyg-lib / torch-scatter 等编译扩展，也不要装它们）
if importlib.util.find_spec("torch_geometric") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric>=2.8,<3"], check=True)
# transformers 在 Colab 已预装；本地环境缺它时兜底安装（课程按 5.x 的 API 编写）
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.22,<6"], check=True)
print("PyTorch Geometric 与 transformers 已就绪")

In [ ]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn.functional as F
from torch import nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore", message=".*torch.jit.script.*")   # 新版 torch 对 PyG 内部写法的弃用提醒，与本课无关
import torch_geometric
from torch_geometric.datasets import KarateClub, Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_dense_adj
import transformers
from transformers import AutoModel, AutoTokenizer, pipeline
transformers.logging.set_verbosity_error()          # 只显示错误：隐藏加载模型时的长篇报告
transformers.utils.logging.disable_progress_bar()

from zero2researcher import data, experiments
from zero2researcher.checks import week13 as ck

# GNN 部分固定用 CPU：本周的图都很小，CPU 足够快；PyG 的部分算子在 Apple 的 MPS 上不受支持
gnn_device = torch.device("cpu")
# 预训练模型部分（S3）自动选择：有 GPU 就用 GPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| torch_geometric", torch_geometric.__version__,
      "| transformers", transformers.__version__)
print("GNN device:", gnn_device, "| transformer device:", device)

---
## S1 · 图与消息传递

### 一张图 = 节点 + 边

**Zachary 空手道俱乐部**（Zachary 1977, *Journal of Anthropological Research* 33(4)）：一所大学空手道俱乐部的 34 名成员，两人在俱乐部之外经常来往，就在他们之间连一条边。研究期间教练 Mr. Hi 与管理员 John A 闹翻，俱乐部一分为二——这张小图从此成了网络科学的 "hello world"。networkx 自带它：

In [ ]:
G = nx.karate_club_graph()
print("节点数:", G.number_of_nodes(), "| 边数:", G.number_of_edges())
print("节点 0（教练 Mr. Hi）的邻居:", sorted(G.neighbors(0)))

PyG 也自带这张图，并且多了两样东西：**节点特征** `x` 与**标签** `y`。注意 PyG 的标签是 **4 个社区**（由基于模块度的聚类得到，Kipf & Welling 附录 A 用的就是这一版），而不是很多教程里讲的"一分为二"；并且只有 **4 个节点**（每个社区 1 个）带训练标签。用 4 个标签猜另外 30 个人属于哪个社区——这就是**半监督节点分类**。

In [ ]:
karate = KarateClub()[0]
print(karate)
print("类别数:", int(karate.y.max()) + 1, "| 每类节点数:", torch.bincount(karate.y).tolist())
print("带训练标签的节点:", karate.train_mask.nonzero().flatten().tolist())
print("edge_index 的前 6 列:\n", karate.edge_index[:, :6])

`edge_index` 是 PyG 存边的方式：一个 (2, 边数) 的整数张量，每一列是一条有向边"源 → 目标"。无向图的每条边存两次（u→v 与 v→u），所以 78 条边变成 156 列。把图画出来（颜色 = 社区，黑框 = 带训练标签的 4 个节点）：

In [ ]:
pos = nx.spring_layout(G, seed=7)
labelled = karate.train_mask.nonzero().flatten().tolist()
fig, ax = plt.subplots(figsize=(6.5, 5))
nx.draw_networkx_edges(G, pos, alpha=0.3, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=karate.y.numpy(), cmap="Set2", node_size=260, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=labelled, node_color="none", edgecolors="black",
                       linewidths=2.5, node_size=380, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set_title("Karate Club: 34 members, 78 ties, 4 communities; only 4 nodes are labelled")
ax.axis("off")
plt.show()

### 邻接矩阵：每一行是一个节点

第 4 周工作坊的约定是"每一行是一个样本"；在图上，**每一行是一个节点**。邻接矩阵 $A$ 是 $34\times 34$ 的 0/1 矩阵：$A_{vu} = 1$ 当且仅当 $v$ 与 $u$ 之间有边。无向图的 $A$ 是对称的；第 $v$ 行的和就是节点 $v$ 的**度**（朋友数）。

In [ ]:
A = torch.tensor(nx.to_numpy_array(G, nodelist=range(34), weight=None), dtype=torch.float32)
print("A 的形状:", tuple(A.shape), "| 对称:", torch.equal(A, A.T), "| 非零元素:", int(A.sum()), "= 2 × 78")
print("与 PyG 的 edge_index 是同一张图:", torch.equal(A, to_dense_adj(karate.edge_index)[0]))
deg = A.sum(dim=1)
print("每个节点的度:", deg.int().tolist())

### 手算一次消息传递

GNN 的核心只有一句话：**每个节点收集邻居的特征，汇总成自己的新特征。** 先给每个节点一个最简单的特征——它的度，$x_v = \deg(v)$。

节点 9 只有两个朋友：节点 2（度 10）与节点 33（度 17）。拿出纸笔：

- 求和聚合：$h_9 = x_2 + x_{33} = 10 + 17 = 27$
- 平均聚合：$h_9 = (10 + 17) / 2 = 13.5$

再用代码对一遍：

In [ ]:
X_deg = deg.unsqueeze(1)                 # (34, 1)：每一行是一个节点，这一列特征是它的度
v = 9
nbrs = A[v].nonzero().flatten()          # A 的第 v 行里等于 1 的位置 = v 的邻居
print("节点 9 的邻居:", nbrs.tolist(), "| 邻居的特征:", X_deg[nbrs].flatten().tolist())
h_sum = X_deg[nbrs].sum(dim=0)
h_mean = X_deg[nbrs].mean(dim=0)
print("求和聚合:", h_sum.item(), "| 平均聚合:", h_mean.item())

对**所有节点同时**做这件事，就是一次矩阵乘法。回忆第 4 周："$A @ B$ 的第 $i$ 行第 $j$ 列 = $A$ 的第 $i$ 行与 $B$ 的第 $j$ 列的点积"。$A$ 的第 $v$ 行只在邻居处为 1，所以

$$(AX)_v = \sum_u A_{vu}\, x_u = \sum_{u \in \mathcal N(v)} x_u$$

恰好是"把邻居的特征加起来"。平均聚合再除以度：$D^{-1}AX$，$D$ 是把度放在对角线上的矩阵。

In [ ]:
H_sum = A @ X_deg                        # 所有节点的求和聚合，一次算完
H_mean = H_sum / deg.unsqueeze(1)        # 等价于 D^{-1} A X
print("矩阵形式，节点 9：求和", H_sum[v].item(), "| 平均", H_mean[v].item())

### GCN 的两处改动：自环与对称归一化

Kipf & Welling 在求和聚合上做了两处改动（论文 Eq. 8，他们称之为 *renormalization trick*）：

1. **加自环** $\tilde A = A + I$：节点自己的特征也参与汇总，否则一层之后它就"忘了自己"。
2. **对称归一化** $\hat A = \tilde D^{-1/2}\,\tilde A\,\tilde D^{-1/2}$，其中 $\tilde D$ 是 $\tilde A$ 的度矩阵。邻居 $u$ 传给 $v$ 的消息变成 $x_u / \sqrt{\tilde d_u \tilde d_v}$——**朋友越多的人，每条消息的分量越轻**；否则一个"社交明星"的特征会被放大几十倍。论文说，renormalization trick 是为了**缓解**（原文 *alleviate*）反复相乘带来的数值不稳定与梯度爆炸/消失——下面会看到 $\hat A$ 的特征值都落在 $(-1, 1]$ 之内。

于是**一层 GCN** 就是

$$H' = \sigma\big(\hat A\, H\, W\big)$$

先用同一个 $W$ 对每个节点做线性变换（就是第 8 周的 `nn.Linear`），再用 $\hat A$ 在邻居之间混合，最后过激活函数 $\sigma$。

手算节点 9（加自环后 $\tilde d_9 = 3,\ \tilde d_2 = 11,\ \tilde d_{33} = 18$；自己的特征是 2）：

$$(\hat A X)_9 = \frac{2}{3} + \frac{10}{\sqrt{3\cdot 11}} + \frac{17}{\sqrt{3\cdot 18}} \approx 0.667 + 1.741 + 2.313 = 4.721$$

In [ ]:
def normalized_adjacency(A):
    # Â = D̃^{-1/2} (A + I) D̃^{-1/2}（Kipf & Welling Eq. 8）；A 是 (n, n) 稠密 0/1 矩阵
    A_tilde = A + torch.eye(len(A))
    d_inv_sqrt = A_tilde.sum(dim=1).pow(-0.5)       # 先算度向量，再取 -1/2 次方
    D_inv_sqrt = torch.diag(d_inv_sqrt)
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt


A_hat = normalized_adjacency(A)
hand = 2 / 3 + 10 / np.sqrt(3 * 11) + 17 / np.sqrt(3 * 18)
print(f"节点 9：手算 {hand:.4f} | 矩阵算 {(A_hat @ X_deg)[v].item():.4f}")
eig = torch.linalg.eigvalsh(A_hat)
print(f"Â 的特征值范围: [{eig.min():.3f}, {eig.max():.3f}]")
ck.normalized_adjacency(normalized_adjacency)

### 纯 PyTorch 的一层 GCN

有了 $\hat A$（只算一次，之后反复用），一层 GCN 只需要一个线性层和一次矩阵乘法。照论文的写法，**层里不放激活**，激活写在模型的 `forward` 里；权重用论文的 Glorot 初始化（就是第 9 周的 Xavier，PyG 的 `GCNConv` 也用它）：

In [ ]:
class GCNLayer(nn.Module):
    """一层 GCN：H' = Â H W。Â 预先算好，作为参数传进 forward。"""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)
        nn.init.xavier_uniform_(self.linear.weight)

    def forward(self, X, A_hat):
        return A_hat @ self.linear(X)             # (n, n) @ (n, out) → (n, out)


ck.gcn_layer(GCNLayer)

### 在 Karate Club 上只用 4 个标签训练

照 Kipf & Welling 附录 A 和 PyG 教程 1 的做法：**不给任何节点特征**（$X = I$，每个节点只知道"我是谁"），叠三层 GCN（34 → 4 → 4 → 2，tanh），最后一层输出 2 维，正好画在平面上；再接一个线性分类器输出 4 个社区。先看**还没训练**时的 2 维嵌入，再用 4 个标签训练 300 步（Adam，lr 0.01，与论文附录 A.2 相同）：

In [ ]:
class KarateGCN(nn.Module):
    # 照 Kipf & Welling 附录 A（Eq. 13）与 PyG 教程 1：三层 GCN + 线性分类器
    def __init__(self):
        super().__init__()
        self.gcn1, self.gcn2, self.gcn3 = GCNLayer(34, 4), GCNLayer(4, 4), GCNLayer(4, 2)
        self.classifier = nn.Linear(2, 4)

    def forward(self, X, A_hat):
        h = torch.tanh(self.gcn1(X, A_hat))
        h = torch.tanh(self.gcn2(h, A_hat))
        h = torch.tanh(self.gcn3(h, A_hat))          # 2 维嵌入：可以直接画出来
        return self.classifier(h), h


X_id = torch.eye(34)                     # 没有节点特征：每个节点是一个 one-hot
train_mask = karate.train_mask


def train_karate(seed, steps=300):
    # 返回 (训练前的 2 维嵌入, 训练后的 2 维嵌入, 30 个无标签节点上的准确率)
    experiments.set_seed(seed)
    model = KarateGCN()
    with torch.no_grad():
        _, emb_before = model(X_id, A_hat)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for step in range(steps):
        logits, _ = model(X_id, A_hat)
        loss = F.cross_entropy(logits[train_mask], karate.y[train_mask])    # 损失只算 4 个有标签的节点
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        logits, emb_after = model(X_id, A_hat)
    acc = (logits.argmax(dim=1) == karate.y)[~train_mask].float().mean().item()
    return emb_before, emb_after, acc


emb_before, emb_after, acc0 = train_karate(seed=0)
print(f"种子 0：另外 30 个没有标签的节点，准确率 {acc0:.3f}")
karate_accs = [train_karate(seed)[2] for seed in range(10)]
print("10 个种子:", np.round(karate_accs, 3), "→", experiments.format_result(karate_accs))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, emb, title in [(axes[0], emb_before, "Untrained GCN (random weights, seed 0)"),
                       (axes[1], emb_after, "After 300 steps with 4 labels (seed 0)")]:
    z = emb.numpy()
    ax.scatter(z[:, 0], z[:, 1], c=karate.y.numpy(), cmap="Set2", s=70, edgecolors="gray")
    ax.scatter(z[labelled, 0], z[labelled, 1], facecolors="none", edgecolors="black", s=160, linewidths=2)
    ax.set_title(title)
    ax.set_xlabel("Embedding dim 1")
    ax.set_ylabel("Embedding dim 2")
fig.suptitle("Even an untrained GCN lines up the communities; 4 labels then push them to corners")
plt.tight_layout()
plt.show()

左图是**随机权重、完全没训练**的 GCN：同一社区的节点已经大体排在一起——三层 GCN 把每个节点 3 跳以内的邻域"搅拌"到一起，而社区内部连接稠密，同一社区的人邻域本来就相似（Kipf & Welling 附录图 3b 展示的就是这个现象）。**结构本身就是信息**。训练之后，4 个标签把各社区推向不同的角落。

但看一眼 10 个种子的结果：只有 4 个标签时，30 个节点的准确率随种子在很大范围内波动。单看某一个种子的漂亮图，很容易高估方法——这正是第 6 周起坚持"至少 3 个种子、报告均值 ± 标准差"的原因。

### 节点重新编号：置换等变性

节点的编号是任意的——把 34 个人重新编号，图还是那张图。我们希望模型的输出**跟着人走**：张三的新编号是 5，那么输出的第 5 行就该是张三原来的输出。

用**置换矩阵** $P$（把单位矩阵的行重新排列）表示重新编号：新的特征是 $PX$，新的邻接矩阵是 $PAP^\top$（行和列都要换）。度也跟着重排，所以 $\hat A$ 变成 $P\hat AP^\top$，于是

$$f(PX,\ P\hat AP^\top) = P\hat AP^\top\, PX\, W = P\,(\hat A X W) = P\, f(X, \hat A)$$

（用到了 $P^\top P = I$）。这叫**置换等变**（equivariant）：输入怎么重排，输出就怎么重排。如果再把所有节点的输出加起来，得到代表"整张图"的一个向量，它就与编号完全无关——**置换不变**（invariant）。用一层随机初始化的 GCN 验证：

In [ ]:
experiments.set_seed(0)
layer = GCNLayer(8, 4)
X_rand = torch.randn(34, 8)              # 随便给 34 个节点各 8 维特征
perm = torch.randperm(34)
P = torch.eye(34)[perm]                  # 置换矩阵：P @ X 把 X 的行按 perm 重新排列
print("先重新编号再归一化 = 先归一化再重新编号:",
      torch.allclose(normalized_adjacency(P @ A @ P.T), P @ A_hat @ P.T, atol=1e-6))
with torch.no_grad():
    out = layer(X_rand, A_hat)
    out_perm = layer(P @ X_rand, P @ A_hat @ P.T)
diff_node = (out_perm - P @ out).abs().max().item()
diff_graph = (out_perm.sum(dim=0) - out.sum(dim=0)).abs().max().item()
diff_naive = (out_perm - out).abs().max().item()
print(f"f(PX, PÂPᵀ) 与 P·f(X, Â) 的最大差异: {diff_node:.1e}   ← 等变")
print(f"图级读出（对所有节点求和）的差异:     {diff_graph:.1e}   ← 不变")
print(f"不把输出重排、直接比较的差异:         {diff_naive:.2f}      ← 输出确实跟着节点换了位置")
ck.permutation_equivariance(diff_node, diff_graph)

三个数说明了一切：先重排再算、先算再重排，结果相同（差异只是浮点误差）；图级读出与编号无关；但输出本身确实跟着节点换了位置——所以叫"等变"而不是"不变"。

为什么这很重要？假如把邻接矩阵的每一行直接喂给一个 MLP，节点 3 的输入"我和 0、1、2 号是朋友"会随编号改变，模型只能从数据里**重新学会**"编号无关"这件事。GCN 把这个对称性**写进了结构**，就像第 10 周的卷积把"平移"写进了结构。第 11 周你还验证过：**没有位置编码的自注意力**同样是置换等变的——自注意力可以看成在"所有 token 两两相连的完全图"上做消息传递，边权由注意力分数给出。

---
## S2 · PyTorch Geometric 与 Cora 节点分类

**Cora** 是一个论文引用网络（Sen et al. 2008；公开划分来自 Yang, Cohen & Salakhutdinov, ICML 2016 的 Planetoid）：2708 篇机器学习论文，每篇属于 7 个主题之一；节点特征是 1433 维的 0/1 词袋（词表中每个词在这篇论文里出现与否）；边是引用关系。任务：只知道 140 篇（每类 20 篇）的主题，预测其余论文的主题——这正是 Kipf & Welling 表 2 的主战场。

PyG 的 `Planetoid` 一行就能下载（从 [kimiyoung/planetoid](https://github.com/kimiyoung/planetoid) 取 8 个原始文件，几秒钟；之后读本地缓存）：

In [ ]:
dataset = Planetoid(root=data.download_root(), name="Cora")
cora = dataset[0].to(gnn_device)
print(cora)
print(f"节点 {cora.num_nodes} | 特征维度 {dataset.num_features} | 类别 {dataset.num_classes}")
# 边数口径：PyG 把每条无向边存成两条有向边（u→v 与 v→u），所以 num_edges = 10556 = 2 × 5278。
# 论文里常见的"5429 条边"是原始数据的引用条数，去重、对称化之后只剩 5278 条无向边（见下面的说明）。
print(f"edge_index {tuple(cora.edge_index.shape)} → 有向边 {cora.num_edges} 条 = 无向边 {cora.num_edges // 2} 条 × 2"
      f" | 是无向图: {cora.is_undirected()}")
print(f"公开划分：训练 {int(cora.train_mask.sum())} | 验证 {int(cora.val_mask.sum())} | 测试 {int(cora.test_mask.sum())}")
print("每类训练节点数:", torch.bincount(cora.y[cora.train_mask]).tolist())
print(f"平均度 {cora.num_edges / cora.num_nodes:.2f} | 每篇论文平均含词表中的 {cora.x.sum(dim=1).mean():.1f} 个词")
ck.cora_stats(cora.num_nodes, dataset.num_features, dataset.num_classes,
              n_edges=cora.num_edges, n_train=int(cora.train_mask.sum()))

**边数口径**——这是复现时你会遇到的第一个"对不上的数"：Kipf & Welling 的表 1（转引 Yang et al. 2016）写 Cora 有 **5,429** 条边，而你在 PyG 里数到的是 **10556**。原因有两层：PyG 把每条无向边存两次，10556 = 2 × 5278；原始数据里的 5429 条引用经过去重、对称化之后只剩 **5278** 条无向边。读论文对数字时，先弄清**口径**。

训练循环照 PyG 教程 2：每一步把**整张图**送进模型（full batch，Cora 很小），但损失只在 140 个训练节点上算；评估时只看测试节点。

In [ ]:
def train_node_classifier(model, data, epochs=200, lr=0.01, weight_decay=5e-4):
    # PyG 教程 2 的训练循环：整张图一次前向，只在 train_mask 的节点上算损失
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for epoch in range(epochs):
        model.train()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model


def accuracy(model, data, mask):
    model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(dim=1)
    return (pred[mask] == data.y[mask]).float().mean().item()

### 基线：只看特征的 MLP

PyG 教程 2 的第一步是一个**完全不看图**的 MLP：每篇论文只凭自己的词袋猜主题。这是本周最重要的基线——GCN 的任何提升都要和它比。

In [ ]:
class MLP(nn.Module):
    # PyG 教程 2 的基线：Linear(1433, 16) → ReLU → Dropout(0.5) → Linear(16, 7)，不看引用关系
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden)
        self.lin2 = nn.Linear(hidden, out_dim)

    def forward(self, x, edge_index=None):        # 收下 edge_index 但不用它，好与 GCN 共用训练函数
        x = F.dropout(self.lin1(x).relu(), p=0.5, training=self.training)
        return self.lin2(x)


experiments.set_seed(0)
mlp = train_node_classifier(MLP(dataset.num_features, 16, dataset.num_classes).to(gnn_device), cora)
mlp_acc = accuracy(mlp, cora, cora.test_mask)
print(f"MLP：训练准确率 {accuracy(mlp, cora, cora.train_mask):.3f} | 测试准确率 {mlp_acc:.3f}")
ck.mlp_cora_accuracy(mlp_acc)

### GCN：特征 + 图

把 `nn.Linear` 换成 PyG 的 `GCNConv`，其余（隐藏层 16、dropout 0.5、Adam lr 0.01、weight decay 5e-4、200 epochs）不变。`GCNConv` 接收 `edge_index` 而不是稠密矩阵，内部自动加自环、做对称归一化。下面把它写成任意层数，讲过平滑时还要用。

**先说清楚：这和论文的设置并不完全一样。** 复现者要能逐条说出差异（S4 会再整理一遍）：

- **特征预处理（最重要的一条）**：Kipf & Welling §5.2 写明对输入特征做了**行归一化**（"(row-)normalize input feature vectors"），PyG 教程 2 也是 `Planetoid(..., transform=NormalizeFeatures())`——每篇论文的词袋向量除以它含的词数。本课**保留原始 0/1 特征**，没有做这一步。这个偏离会**改变层数消融的结论**：本机 3 个种子下，原始特征的 8 层 GCN 约 0.62，仍高于 MLP 的 0.55；而在论文的行归一化下，8 层 GCN 只有 0.15–0.41，连训练集都学不会（训练准确率 0.19–0.60），**低于 MLP**。所以下文"8 层仍比 MLP 好"只在原始特征下成立。我们保留这个偏离，是因为"同一个结论在两种预处理下相反"本身就是一个值得研究的问题——作业 A 会请你亲手验证并解释它。
- 论文的 weight decay 只加在第一层，我们加在全部参数上；论文用验证集早停（窗口 10），我们不早停；论文报告 100 次随机初始化的平均，我们只跑 1 个种子。

In [ ]:
class GCN(nn.Module):
    # PyG 教程 2 的 GCN 推广到任意层数：第一层 1433 → 16，中间层 16 → 16，最后一层 16 → 7
    def __init__(self, in_dim, hidden, out_dim, num_layers=2):
        super().__init__()
        dims = [in_dim] + [hidden] * (num_layers - 1) + [out_dim]
        self.convs = nn.ModuleList([GCNConv(a, b) for a, b in zip(dims[:-1], dims[1:])])

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:               # 最后一层输出 logits，不加激活和 dropout
                x = F.dropout(x.relu(), p=0.5, training=self.training)
        return x


experiments.set_seed(0)
gcn = train_node_classifier(GCN(dataset.num_features, 16, dataset.num_classes).to(gnn_device), cora)
gcn_acc = accuracy(gcn, cora, cora.test_mask)
print(f"GCN：训练准确率 {accuracy(gcn, cora, cora.train_mask):.3f} | 测试准确率 {gcn_acc:.3f}")
ck.gcn_cora_accuracy(gcn_acc, model=gcn, data=cora)     # 传入模型：自查会在 eval 模式下重算一次对账

`GCNConv` 和我们在 S1 写的 `GCNLayer` 是同一个东西吗？把 Cora 的 `edge_index` 转成稠密的 $2708 \times 2708$ 邻接矩阵（约 29MB，Cora 还放得下；大图上 PyG 坚持用稀疏运算，就是为了避免这一步），再把训练好的第一层权重抄进我们的层：

In [ ]:
A_cora = to_dense_adj(cora.edge_index, max_num_nodes=cora.num_nodes)[0]
A_hat_cora = normalized_adjacency(A_cora)
conv = gcn.convs[0]
mine = GCNLayer(dataset.num_features, 16)
with torch.no_grad():
    mine.linear.weight.copy_(conv.lin.weight)          # GCNConv 内部也是一个不带 bias 的线性层，bias 单独加
    diff = (conv(cora.x, cora.edge_index) - (mine(cora.x, A_hat_cora) + conv.bias)).abs().max().item()
print(f"PyG GCNConv 与纯 PyTorch GCNLayer（同一组权重）的最大差异: {diff:.1e}")

In [ ]:
names = ["MLP\n(features only)", "2-layer GCN\n(features + graph)"]
ours, paper = [mlp_acc, gcn_acc], [0.551, 0.815]          # 论文值：Kipf & Welling 表 3 的 MLP、表 2 的 GCN
fig, ax = plt.subplots(figsize=(5.5, 3.6))
bars = ax.bar(names, ours, color=["gray", "C0"], label="This notebook (seed 0)")
ax.bar_label(bars, fmt="%.3f", label_type="center", color="white", fontweight="bold")
ax.scatter(names, paper, color="black", marker="_", s=900, zorder=3, label="Kipf & Welling 2017")
ax.set_ylim(0, 1)
ax.set_ylabel("Test accuracy (1000 nodes)")
ax.set_title("Using the citation graph lifts accuracy from ~0.54 to ~0.81")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

同样 140 个标签，MLP 约 0.54，GCN 约 0.81——**近 28 个百分点全部来自图结构**。原因是 Cora 的引用关系有很强的**同质性**（homophily）：论文倾向于引用同主题的论文（练习 E3 你会量出它有多强）。GCN 让每篇论文"借用"邻居的词袋，140 个标签的信息就沿着引用边扩散到了整张图。MLP 的训练准确率是 1.0、测试只有 0.54：140 个样本、1433 维特征，它只能死记硬背。

两个数字都和论文对得上：Kipf & Welling 表 2 的 GCN 是 81.5%（100 次随机初始化的平均），表 3 的 MLP 是 55.1%。

### 层数越多越好吗？过平滑

一层 GCN 让每个节点看到 1 跳邻居，$K$ 层看到 $K$ 跳——就像 CNN 越深感受野越大。那么 8 层是不是更好？只改层数，其余不变：

In [ ]:
experiments.set_seed(0)
gcn8 = train_node_classifier(GCN(dataset.num_features, 16, dataset.num_classes, num_layers=8).to(gnn_device), cora)
acc8 = accuracy(gcn8, cora, cora.test_mask)
print(f"2 层 GCN：测试 {gcn_acc:.3f} | 8 层 GCN：测试 {acc8:.3f}，训练 {accuracy(gcn8, cora, cora.train_mask):.3f}")
ck.oversmoothing(gcn_acc, acc8)

8 层反而掉了约 19 个百分点。为了看清原因，把可学习的部分全部拿掉，只做"反复乘 $\hat A$"：$H^{(k)} = \hat A^k X$。看两件事：① 节点表示两两之间的平均余弦相似度（越接近 1，所有节点越像）；② 在 $H^{(k)}$ 上训练一个逻辑回归（同样只用 140 个训练节点）的测试准确率。$\hat A$ 只有约 1.3 万个非零元素，用稀疏矩阵乘法很快：

In [ ]:
A_hat_sparse = A_hat_cora.to_sparse()
train_idx, test_idx = cora.train_mask.numpy(), cora.test_mask.numpy()
y_np = cora.y.numpy()
ks, mean_cos, lr_acc = [0, 1, 2, 4, 8, 16, 32, 64], [], []
H = cora.x.clone()
for k in range(max(ks) + 1):
    if k in ks:
        Hn = F.normalize(H, dim=1)                       # 每一行除以自己的长度
        mean_cos.append((Hn @ Hn.T).mean().item())       # 两两余弦相似度的平均
        clf = LogisticRegression(max_iter=1000).fit(H[train_idx].numpy(), y_np[train_idx])
        lr_acc.append(clf.score(H[test_idx].numpy(), y_np[test_idx]))
        print(f"k = {k:2d}: 平均余弦相似度 {mean_cos[-1]:.3f} | 逻辑回归测试准确率 {lr_acc[-1]:.3f}")
    H = A_hat_sparse @ H                                 # 再传播一次

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
pos_k = np.arange(len(ks))                               # 横轴按"翻倍"等间距排列
axes[0].plot(pos_k, mean_cos, "o-")
axes[0].set_ylabel("Mean pairwise cosine similarity")
axes[0].set_title("Nodes look more and more alike")
axes[1].plot(pos_k, lr_acc, "o-", color="C1")
axes[1].axhline(lr_acc[0], color="gray", ls="--", label="k = 0 (features only)")
axes[1].set_ylabel("Logistic regression test accuracy")
axes[1].set_title("Accuracy peaks around k = 2-4, then falls")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xticks(pos_k, [str(k) for k in ks])
    ax.set_xlabel("Propagation steps k  (H = Â^k X, no learning)")
fig.suptitle("Over-smoothing: repeated neighbour averaging washes out the class signal")
plt.tight_layout()
plt.show()

传播 1–2 次，逻辑回归就从 0.58 跳到 0.82 以上（峰值在 k = 2–4）——几乎不需要学习，"把邻居的特征平均进来"本身就是 GCN 赢 MLP 的原因。之后每多传播一次，所有节点就更像彼此一点：传播 64 次后，平均余弦相似度超过 0.8，准确率跌回 0.64。**反复求平均会把所有节点抹成同一种颜色**——这叫**过平滑**（over-smoothing；Li, Han & Wu, AAAI 2018 最早系统地分析了它）。数学上：$\hat A$ 的最大特征值是 1，反复相乘之后，特征被推向特征值 1 的特征向量张成的子空间。Cora 并不连通——它有 78 个连通分量（最大的一个有 2485 个节点），所以特征值 1 是 78 重的；极限是**在每个连通分量内部**都正比于 $\tilde D^{1/2}\mathbf 1$（Li, Han & Wu 的原话是收敛到 "the linear combination of eigenvectors of eigenvalue 1"）。也就是说，节点的表示最终只由它的**度和所在的连通分量**决定，与主题无关。

Kipf & Welling 附录 B 的图 5 做的正是层数消融（1–10 层）：2–3 层最好，超过 7 层时不加残差连接就很难训练。8 层 GCN 的问题也不只有过平滑：看上一格，它连**训练**准确率都到不了 1.0——更深的网络也更难优化。两种解释各占多少？这是你作业 A 要讨论的问题之一。（别忘了上面说的预处理差异：这一段的数字都来自原始 0/1 特征。）

### （选做）看一看节点嵌入

把 MLP 和 GCN 对全部 2708 个节点的输出（7 维 logits）用 t-SNE 降到 2 维，按真实主题上色。第 6 周的提醒依然有效：**t-SNE 图里的距离和簇的大小都不可解读**，只看"同色的点是否聚在一起"。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (name, model) in zip(axes, [("MLP", mlp), ("2-layer GCN", gcn)]):
    model.eval()
    with torch.no_grad():
        out = model(cora.x, cora.edge_index)
    z = TSNE(n_components=2, random_state=0).fit_transform(out.numpy())
    ax.scatter(z[:, 0], z[:, 1], c=y_np, cmap="tab10", s=3)
    ax.set_title(f"{name} outputs (t-SNE)")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("GCN outputs form cleaner topic clusters than MLP outputs (positions are not to scale)")
plt.tight_layout()
plt.show()

---
## S3 · 预训练与迁移学习

到目前为止，你的每一个模型都从随机初始化开始、只在本任务的数据上训练。**预训练**换了一种思路：先在海量**无标签**数据上用一个"自监督"目标训练一个通用模型，再把它**迁移**到你的小任务上。标签从数据自己身上来——不需要任何人工标注。文本上最常见的两种目标：

| | **掩码语言模型 MLM**（BERT） | **自回归语言模型**（GPT） |
| --- | --- | --- |
| 训练目标 | 随机遮住约 15% 的 token，用**两侧**上下文预测被遮住的词 | 根据**前面**所有 token 预测下一个 token |
| 损失 | $-\sum_{i \in \text{masked}} \log p(x_i \mid x_{\setminus \text{masked}})$ | $-\sum_t \log p(x_t \mid x_{<t})$ |
| 注意力 | 双向（每个位置看得到左右两边） | 因果掩码（只看左边，第 11 周） |
| 擅长 | 理解：分类、抽取、句子表示 | 生成：续写、对话 |
| 本课程 | 本周的 DistilBERT | 第 12 周你训练的字符级 GPT |

BERT 出自 Devlin, Chang, Lee & Toutanova 2018（[arXiv:1810.04805](https://arxiv.org/abs/1810.04805)）；GPT 出自 Radford et al. 2018（OpenAI 技术报告 *Improving Language Understanding by Generative Pre-Training*）。本周用 **DistilBERT**（Sanh et al. 2019, [arXiv:1910.01108](https://arxiv.org/abs/1910.01108)；模型 `distilbert-base-uncased`，Apache-2.0）：用知识蒸馏把 BERT 压缩，摘要原话是"reduce the size of a BERT model by 40%, while retaining 97% of its language understanding capabilities and being 60% faster"。

DistilBERT 正是用 MLM 训练的——让它做一下"完形填空"（首次运行会从 Hugging Face 下载约 268MB 的模型，之后读缓存）：

In [ ]:
fill_mask = pipeline("fill-mask", model="distilbert-base-uncased", device=device)
for sentence in ["Paris is the [MASK] of France.",
                 "The movie was absolutely [MASK].",
                 "This film is a [MASK] of time."]:
    preds = fill_mask(sentence, top_k=5)
    print(f"{sentence:34s} → " + ", ".join(f"{p['token_str']} ({p['score']:.2f})" for p in preds))

没有人告诉过它"这是一条影评"，但它知道 "absolutely ___" 后面多半是一个强烈的褒义形容词，"a ___ of time" 最可能是 *waste*。为了猜对被遮住的词，模型必须学会语法、词义乃至常识——这些副产品，就是我们想迁移的**表示**。

### 基础模型一览

在海量数据上预训练、再迁移到许多下游任务的大模型，今天统称**基础模型**（foundation models）。四个最常被提到的代表：

| 模型 | 数据 / 模态 | 预训练目标 | 典型用法 |
| --- | --- | --- | --- |
| BERT（Devlin et al. 2018） | 文本（BooksCorpus + 英文维基百科） | MLM（外加"下一句预测"） | 微调做分类、抽取 |
| GPT（Radford et al. 2018） | 文本（BooksCorpus） | 自回归 | 生成；后继者 GPT-2/3 发展出提示与少样本学习 |
| ViT（Dosovitskiy et al. 2020, [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)） | 图像：切成 16×16 的小块，每块当一个 token | 在大规模有标签图像数据上做监督预训练 | 图像分类的骨干网络 |
| CLIP（Radford et al. 2021, [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)） | 约 4 亿个（图像，文字说明）对 | 对比学习：让图像与它对应文字的嵌入靠近 | 零样本图像分类、图文检索 |

它们的共同点：**Transformer 骨干 + 大规模数据 + 预训练一次、迁移到许多任务**。其中 BERT、GPT、CLIP 的训练信号都来自数据本身（被遮住的词、下一个词、图片配的文字），不需要逐条人工标注；ViT 原文则是在大规模有标签图像上预训练的。

### 冻结的 DistilBERT 当特征提取器：线性探针

最简单的迁移方式：**冻结**预训练模型（一个参数都不改），把每条影评送进去，取最后一层的隐藏状态 (B, T, 768)，池化成每条影评一个 768 维向量，再在上面训练逻辑回归。只训练一个线性分类器，所以叫**线性探针**（linear probe）——它衡量的是：预训练表示里，"好评/差评"已经有多么线性可分。

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"   # 固定模型版本（Hub 上的提交号）：模型仓库以后更新，课程里的数字也能复现
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
bert = AutoModel.from_pretrained(MODEL_NAME, revision=REVISION).to(device).eval()
print(f"DistilBERT 参数量: {sum(p.numel() for p in bert.parameters()) / 1e6:.1f}M")

batch = tokenizer(["A great movie!", "A terrible, boring movie with no plot."],
                  padding=True, truncation=True, max_length=256, return_tensors="pt")
print("第 1 句的 token:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0]))
print("attention_mask:\n", batch["attention_mask"])
with torch.no_grad():
    out = bert(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
print("last_hidden_state:", tuple(out.last_hidden_state.shape), "= (句子数, token 数, 768)")

两件事要看清：

1. 分词器在句首加 `[CLS]`、句尾加 `[SEP]`；一个 batch 里短句子用 `[PAD]` 补到一样长，`attention_mask` 里真实 token 是 1、padding 是 0。
2. 每个 token 都得到一个 768 维向量。怎样把一句话变成**一个**向量？两种常见做法：
   - **[CLS] 池化**：取第一个位置（`[CLS]`）的向量；
   - **平均池化**（mean-pool）：对所有**真实** token 的向量求平均——padding 位置必须排除，否则句子越短，被 `[PAD]` 的向量"稀释"得越厉害。

$$\text{mean-pool}(h) = \frac{\sum_t m_t\, h_t}{\sum_t m_t}, \qquad m_t \in \{0, 1\} \text{ 是 attention\_mask}$$

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    # (B, T, H) 与 (B, T) → (B, H)：只平均 attention_mask 为 1 的位置
    m = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)       # (B, T, 1)，才能与 (B, T, H) 相乘
    return (last_hidden_state * m).sum(dim=1) / m.sum(dim=1)


ck.mean_pool(mean_pool)

In [ ]:
def embed(texts, batch_size=32, max_length=256):
    """冻结的 DistilBERT：返回 ([CLS] 嵌入, mean-pool 嵌入)，各是 (N, 768) 的 numpy 数组。"""
    cls_out, mean_out = [], []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(list(texts[i:i + batch_size]), padding=True, truncation=True,
                          max_length=max_length, return_tensors="pt").to(device)
        with torch.no_grad():                            # 冻结：不需要梯度
            h = bert(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state
        cls_out.append(h[:, 0].cpu().numpy())
        mean_out.append(mean_pool(h, batch["attention_mask"]).cpu().numpy())
    return np.concatenate(cls_out), np.concatenate(mean_out)


imdb_train, imdb_test = data.load_imdb("train"), data.load_imdb("test")
n_demo = 500                                             # 讲义只嵌入前 500 条训练 + 前 500 条测试
t0 = time.time()
tr_cls, tr_mean = embed(imdb_train["text"][:n_demo])
te_cls, te_mean = embed(imdb_test["text"][:n_demo])
y_tr, y_te = imdb_train["label"][:n_demo].values, imdb_test["label"][:n_demo].values
print(f"嵌入 {2 * n_demo} 条影评用时 {time.time() - t0:.0f} 秒（{device}）| 训练嵌入的形状 {tr_mean.shape}")

在 500 条嵌入上训练逻辑回归，与**同样 500 条**训练数据上的 TF-IDF + 逻辑回归（第 6 周的基线）比——公平的比较要让两边看到同样多的标签：

In [ ]:
probe = {}
for name, (X_a, X_b) in {"[CLS]": (tr_cls, te_cls), "mean-pool": (tr_mean, te_mean)}.items():
    clf = LogisticRegression(max_iter=1000).fit(X_a, y_tr)
    probe[name] = clf.score(X_b, y_te)
tfidf = TfidfVectorizer(min_df=2)
tfidf_clf = LogisticRegression(max_iter=1000).fit(tfidf.fit_transform(imdb_train["text"][:n_demo]), y_tr)
probe["TF-IDF"] = tfidf_clf.score(tfidf.transform(imdb_test["text"][:n_demo]), y_te)
for name, acc in probe.items():
    print(f"{name:10s} + 逻辑回归（训练 {n_demo} 条，测试 {n_demo} 条）: {acc:.3f}")
ck.probe_accuracy(probe["mean-pool"])

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.4))
bars = ax.bar(list(probe), list(probe.values()), color=["C0", "C0", "gray"])
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0.5, 1.0)
ax.set_ylabel(f"Test accuracy ({n_demo} reviews)")
ax.set_title(f"With {n_demo} training reviews, frozen DistilBERT beats TF-IDF")
plt.tight_layout()
plt.show()

只有 500 条训练数据时，冻结的 DistilBERT 嵌入比 TF-IDF 高约 4–5 个百分点：TF-IDF 只认识这 500 条里出现过的词，而 DistilBERT 在预训练时见过海量文本，知道 *superb* 和 *excellent* 意思相近。

那么用上全部 5000 条训练数据呢？预训练表示的优势会保持、缩小，还是反转？这正是你作业 B 的研究问题——**先在作业里写下你的预测（预注册），再去跑全量**。到时把结果填进这张跨周对比表：

| 周 | 模型 | IMDB 测试准确率 |
| --- | --- | --- |
| 6 | TF-IDF + 逻辑回归 | 0.8665 |
| 10 | TextCNN（从零训练） | 用你自己的数 |
| 11 | 小 Transformer（从零训练） | 用你自己的数 |
| 13 | 冻结 DistilBERT + 线性探针 | 作业 B 揭晓 |

想一想：一个从没见过"影评情感"这个任务的通用表示，配上一个线性分类器，数据多了以后还能保持优势吗？如果不能，要怎样让预训练模型**为这个任务调整自己**？这是第 14 周的主题。

---
## S4 · 论文阅读与复现启动

### 三遍阅读法

本节依据 S. Keshav, "How to Read a Paper", *ACM SIGCOMM Computer Communication Review* 37(3), 2007（篇幅很短，多所大学的课程页都有公开 PDF）。核心主张：**不要从第一个字读到最后一个字**，而是分三遍，每一遍目标不同，而且每一遍之后都可以决定停下来。

| 遍 | 时间 | 读什么 | 读完要能回答 |
| --- | --- | --- | --- |
| 第一遍：鸟瞰 | 5–10 分钟 | 标题、摘要、引言；各节标题；结论；扫一眼参考文献 | **5C**：Category（哪类论文）、Context（接着谁做）、Correctness（假设可信吗）、Contributions（主要贡献）、Clarity（写得清楚吗）——以及"要不要读第二遍" |
| 第二遍：抓内容 | 约 1 小时 | 正文，**认真看每一张图表**（坐标轴、误差棒、几次运行）；跳过证明 | 能把论文的主线讲给一个没读过的人听；列出没读懂的术语与要追的参考文献 |
| 第三遍：重做 | 数小时 | 带着"如果我来做"的问题逐句读 | 隐含假设、缺失的细节、弱点、可做的后续——**复现者必须读到这一遍** |

### 用定位表读一篇我们刚用过的论文

工作表的核心是一张**定位表**：每一项都写清"在哪里"（节号、图表号）和"一句话"（自己的话）。模板在 `templates/paper_reading_worksheet.md`。拿今天 S1–S2 一直在用的 Kipf & Welling 2017 演示一遍：

| 要素 | 位置 | 一句话 |
| --- | --- | --- |
| 问题 | §1 引言 | 图上的半监督节点分类：只有少数节点有标签，怎样利用图结构把标签信息传出去 |
| 主张 | 摘要；§6 | 一阶近似的图卷积在引用网络和知识图谱上"outperforms related methods by a significant margin"，且计算量与边数成线性 |
| 方法 | §2（Eq. 8）、§3.1（Eq. 9） | $\hat A = \tilde D^{-1/2}\tilde A\tilde D^{-1/2}$；两层模型 $Z = \mathrm{softmax}(\hat A\,\mathrm{ReLU}(\hat A X W^{(0)})\,W^{(1)})$ |
| 基线 | §5.3；表 2 | 标签传播 LP、SemiEmb、ManiReg、DeepWalk、ICA、Planetoid——除 ICA 外基线数字都转引自 Planetoid 论文 |
| 消融 | §6.2；表 3 | 只换传播模型（Chebyshev K = 2/3、一阶、单参数、只留一阶项、MLP）：renormalization trick 最好（Cora 81.5），MLP 只有 55.1 |
| 数据与划分 | §5.1–5.2；表 1 | 沿用 Yang et al. 2016 的划分：每类 20 个训练标签，另取 500 个验证、1000 个测试；表 1 写 Cora 有 5,429 条边 |
| 关键图/表 | §6.1 表 2 | Cora 81.5%（100 次随机初始化的平均）；10 次随机划分时 80.1 ± 0.5（原文注明 ± 是**标准误**，不是本课一贯用的标准差） |
| 评价指标与训练配置 | §5.2、§6.1 | 测试准确率；两层、16 个隐单元、dropout 0.5、L2 5e-4（只加在第一层）、Adam lr 0.01、最多 200 epochs、早停窗口 10、Glorot 初始化、输入特征**行归一化** |

第三遍读者（复现者）会注意到的差异——**我们今天的实现与论文并不完全一样**：输入特征没有行归一化（论文 §5.2 做了）；weight decay 加在了所有参数上（论文只加第一层）；没有早停；论文报告 100 次运行的平均，我们只跑了 1 个种子；表 1 的边数与 PyG 的口径不同。2 层 GCN 的数字仍然对得上（0.815 vs 81.5），但你要能说出**哪些差异可能改变结论**——比如第一条就会改变层数消融里"8 层是否仍好于 MLP"的结论（S2 已经提过，作业 A7 请你实测）。这就是复现。

### 复现任务：三篇候选论文

从第 13 周到第 15 周，你要**复现经典论文中的一张图或一个表**——这是从"做作业"到"做研究"的过渡。下面三篇的图号与配置已由课程作者预先核对过；选一篇：

| | ① Adam | ② Dropout | ③ ResNet（简化版） |
| --- | --- | --- | --- |
| 论文 | Kingma & Ba, ICLR 2015（[arXiv:1412.6980](https://arxiv.org/abs/1412.6980)） | Srivastava et al., JMLR 15(56):1929–1958, 2014 | He, Zhang, Ren & Sun, CVPR 2016（[arXiv:1512.03385](https://arxiv.org/abs/1512.03385)） |
| 复现哪张图/表 | **图 2(a)** "MNIST Multilayer Neural Network + dropout"：AdaGrad、RMSProp、SGDNesterov、AdaDelta、Adam 五条 **training cost** 曲线（对数纵轴，0–200 epochs） | **表 2**（MNIST）：Standard NN（Simard 2003，logistic，2×800，无 dropout）**1.60%** → Dropout NN（logistic，3×1024）**1.35%** → ReLU 3×1024 **1.25%** → 再加 max-norm **1.06%**；或**图 4**：多种架构有/无 dropout 的测试误差两簇曲线 | **图 6 左**的简化：plain-20 vs plain-56 的**训练误差**曲线（更深的普通网络训练误差反而更高——"退化"）；有余力再加图 6 中的 ResNet-20/56 |
| 原文配置 | §6.2：两个全连接隐层、各 1000 个 ReLU 单元，minibatch 128，带 dropout（dropout 比例未给出）；§6 开头：学习率、动量等超参数在密集网格上搜索，报告最佳设置（网格取值未给出） | 附录 B.1（MNIST）：图 4 是 2/3/4 个隐层 × 1024/2048 单元共 6 种架构；所有架构（含表 2 各行）隐层保留率 p = 0.5、输入层 p = 0.8，**最终动量 0.95**，所有层 **max-norm 约束 c = 2**；从 6 万张训练图中留 1 万张作验证集调超参数（以 100 万次权重更新为准），再合并训练 100 万次权重更新、不早停；学习率及其日程未给出（附录 A.2 只给经验：dropout 网络的学习率取标准网络的 10–100 倍、动量 0.95–0.99）。注意附录 A.3 的"c 通常取 3 到 4"是通用经验，不是本实验的配置 | §4.2：输入减去逐像素均值（per-pixel mean subtracted）；6n+2 层（n = 3/9 → 20/56 层）、滤波器 {16, 32, 64}、stride 2 卷积降采样、全局平均池化 + 10 路全连接；**捷径全部用恒等映射（option A：升维时补零，不引入参数）**，所以残差网络与对应 plain 网络的层数、宽度、参数量完全相同（表 6：ResNet-20 为 0.27M）——用 1×1 投影捷径参数量就对不上；BN、无 dropout；SGD momentum 0.9、weight decay 1e-4、batch 128；lr 0.1，在 32k/48k 步除以 10，共 64k 步（约 164 epochs，按 45k/5k 划分定出）；pad-4 随机裁剪 + 水平翻转 |
| Colab 可行性 | 可行：Colab T4 上单个优化器 200 epochs 约 10–15 分钟（**估计，尚未实测**）；五条全跑建议缩到 50 epochs——原图前 20 个 epoch 五条曲线还挤在一起，Adam 的优势约 25 epochs 之后才分明，所以别少于 50 | 可行但须缩短：784-1024-1024-10 全量 MNIST 约 50 epochs，有/无 dropout 两组约 15–20 分钟（**估计，尚未实测**） | 原配置不可行（4 个模型 × 64k 步要数小时，估计）；简化版每个模型 10–15 epochs，T4 上单模型约 5–8 分钟，只跑 plain-20/56 一对约 30 分钟（**估计，尚未实测**） |
| 验收口径 | **趋势**，照原文 §6.2 对图 2 的结论 *"Adam shows better convergence than other methods"*：Adam 的 training cost 从约第 25 个 epoch 起一直是五条里最低的，差距越拉越大（原图 200 epochs 时约 7×10⁻³，次低的 SGDNesterov 约 1.1×10⁻²）；前 20 个 epoch 五条曲线挤在一起，RMSProp 与 Adam 交替领先。按上面建议的 50 epochs，原图的次序是 Adam（≈0.022）< RMSProp（≈0.027）< SGDNesterov（≈0.030）≲ AdaGrad（≈0.033）< AdaDelta（≈0.038）；全程到 200 epochs 时变成 Adam < SGDNesterov < AdaDelta ≈ RMSProp < AdaGrad（AdaGrad 约 70 epochs 之后才成为最高的一条）。以上是从原文 PDF 的矢量图读出的约数，论文没给数值。**验收**：3 个种子上，Adam 在最后一个 epoch（≥ 50）的 training cost 都明显低于其余四条 | **数值**（三篇里唯一能逼近原文数字的）：缩短训练下预期无 dropout 约 1.6–1.8%、有 dropout 约 1.2–1.4%（**估计，尚未实测**），重点比较**相对差距** | **趋势**：plain-56 的训练误差高于 plain-20；缩短训练后达不到表 6 的 8.75%（ResNet-20）/ 6.97%（ResNet-56） |
| 主要的坑 | ① 每条曲线要粗搜学习率（这本身就是教学点）；② 纵轴是**训练** cost、对数刻度，不是测试准确率；③ dropout 开关要与子图一致；④ **别选图 2(b)**：它要对比的 SFO 优化器没有现成实现；⑤ 别把图 1 的结论当成图 2 的："Adam 与 SGDNesterov 相近、都快于 AdaGrad"是 §6.1 对**图 1 左**（MNIST 逻辑回归）的描述。基础较弱可改选图 1 左（三条曲线，验收口径就用 §6.1 这句） | ① 原文**没有**"784-800-800-10 + dropout"这一行——1.60% 是引自 Simard 的无 dropout 基线，dropout 行从 3×1024 起；建议统一用 784-1024-1024-10 ReLU 有/无 dropout 两条，向图 4 靠拢；② 输入层 dropout 容易漏；③ 省略 max-norm（c = 2）与高动量（最终 0.95）时绝对值会偏高；④ 训练太短时 dropout 可能反而不占优 | ① 看**训练误差**（图中虚线），不是测试准确率；② 捷径用恒等映射（option A，升维时补零），不要用 1×1 投影；参照 [akamaster/pytorch_resnet_cifar10](https://github.com/akamaster/pytorch_resnet_cifar10)（BSD-2-Clause，复现值 ResNet-20 8.27%、ResNet-56 6.61%）的**架构**，不要照抄它按 200 epochs 设计的训练脚本；③ plain-56 起步可能不收敛，可借原文给 110 层用的预热技巧（先用 lr 0.01）；④ CIFAR-10 下载偶尔不稳 |

### 复现的流程与本周要交的东西

1. **先读，再计划，最后才写代码**。用工作表把论文读到第三遍，尤其要把"原文配置"逐项抄出来——原文没写的，明确写"未给出"。
2. **复现计划**（模板 `templates/replication_plan.md`）：复现哪张图、原文设置、你的设置与每一处差异、数据、**预期数值**、**复现的是趋势还是数值**、时间预算、最大风险。先写下验收标准再动手——跑完才定的标准不叫标准。
3. 时间线：**第 13 周**选题，写完阅读工作表与复现计划（本周作业 C、D）→ **第 14 周** S4 进度自查（至少跑出第一条曲线）→ **第 15 周** S4 写复现报告。

复现没成功也是合法的结果——只要你能说清：差异来自哪里，是你的实现有问题，还是原文结论在你的设置下不成立。

### 本周小结

- 图 = 节点 + 边；邻接矩阵的每一行是一个节点，`edge_index` 把每条无向边存两次
- 消息传递：每个节点汇总邻居的特征；对所有节点同时做 = 一次矩阵乘法 $AX$
- GCN：$\hat A = \tilde D^{-1/2}(A+I)\tilde D^{-1/2}$，一层 $H' = \sigma(\hat A H W)$；它对节点重新编号是**等变**的，图级读出是**不变**的
- Cora 上：只看特征的 MLP ≈ 0.54，2 层 GCN ≈ 0.81——提升全部来自图结构（同质性）；层数太多会**过平滑**（并且更难训练）；深层的结论对特征预处理敏感（论文做了行归一化，我们没做）
- 预训练目标：MLM（双向、填空，BERT）vs 自回归（单向、续写，GPT）；冻结嵌入 + 逻辑回归 = 线性探针
- 冻结的 DistilBERT 在小数据（500 条）时明显赢 TF-IDF；数据多了还赢不赢，是作业 B 的研究问题；第 14 周微调
- 三遍阅读法与定位表；复现计划先写验收标准，并分清"复现趋势"还是"复现数值"